In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/home/mh/kmh/sionna-rt/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /home/mh/kmh/sionna-rt/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a             

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/home/mh/kmh/sionna-rt/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

# 5) Center Frequency Setting 
scene.frequency = 6.0e9

[설정] 경로가 수정된 임시 XML 생성: /home/mh/kmh/sionna-rt/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [4]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [5]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [6]:
import plotly.graph_objects as go


x_vals = road_positions[:, 0]
z_vals = road_positions[:, 2] 
indices = list(range(len(road_positions)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals, y=z_vals,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Z: %{y:.1f}<extra></extra>' # 라벨도 Z로 표기
))

fig.update_layout(
    title="도로 점 확인용 지도 (X - Z 평면)",
    xaxis_title="X Axis (East/West)",
    yaxis_title="y Axis (North/South)", # Y축 라벨을 Z축으로 변경
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

In [7]:
import numpy as np
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# ==============================================================================
# 0. 데이터 준비 & 이동 경로 계산
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [ 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

# 거리 및 시간 계산 (60km/h)
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)

cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]

speed_ms = 60.0 / 3.6
total_time = total_distance / speed_ms
delta_t = 0.05 

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    return pos  

time_steps = np.arange(0, total_time + delta_t, delta_t)

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

scene.tx_array = PlanarArray(num_rows=8, num_cols=8, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [-50.472, 36.869, -181.453], [0, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

start_pos = get_pos_at_time(0.0)
rx = Receiver(name="rx_car", position=start_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 인터랙티브 위젯 (수정완료)
# ==============================================================================
output_widget = widgets.Output()

def update_simulation(frame_idx):
    global paths_last

    t = time_steps[frame_idx]
    current_pos = get_pos_at_time(t)

    rx.position = current_pos
    for name in tx_names:
        scene.transmitters[name].look_at(current_pos)

    paths_last = solver(scene, max_depth=3, samples_per_src=100000,
                        diffuse_reflection=True, diffraction=True)

    with output_widget:
        output_widget.clear_output(wait=True)
        scene.preview(paths=paths_last, show_devices=True, resolution=[800, 600])
        print(f"t={t:.2f}, pos={current_pos}")


slider = widgets.IntSlider(
    value=0, min=0, max=len(time_steps)-1, step=1,
    description='Time Step:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(update_simulation, {'frame_idx': slider})

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

2026-02-23 16:14:29.201454: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 16:14:29.214002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771830869.228752  343000 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771830869.233384  343000 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771830869.244979  343000 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

▼ 슬라이더를 움직여보세요.


IntSlider(value=0, description='Time Step:', layout=Layout(width='600px'), max=1001)

Output()

1 : Rx(수신기) 1개
2 : Rx 포트(편파 VH라서 2개)
3 : Tx 3개
128 : Tx 포트(8×8=64 소자 × VH 2편파 = 128)
9 : 경로 개수(현재 프레임에서 찾은 multipath 개수)
1 : time step(지금은 스냅샷 1개)

In [8]:
frame_idx = 0  # 보고 싶은 프레임 인덱스(원하는 값으로 바꿔)
t = time_steps[frame_idx]
current_pos = get_pos_at_time(t)

# 위치/방향 업데이트
rx.position = current_pos
for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True,synthetic_array=True)

# CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)


t = 0.0
a shape: (1, 2, 3, 128, 6, 1)
tau shape: (1, 3, 6)


I0000 00:00:1771830873.056524  343000 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20144 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6


In [9]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# 계산 결과 캐시: (frame_idx, tx_idx, rel_delay) -> (t, pos, df)
_pdp_cache = {}

out = widgets.Output()

tx_dropdown = widgets.Dropdown(
    options=[("Tx_1", 0), ("Tx_2", 1), ("Tx_3", 2)],
    value=0,
    description="TX:",
    layout=widgets.Layout(width="200px")
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_steps)-1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

# 0부터 시작하는 상대지연으로 볼지 옵션 (원하면 켜기)
rel_delay_chk = widgets.Checkbox(
    value=False,
    description="Relative delay (min τ = 0)",
    indent=False
)

def _compute_mapping_for_frame(frame_idx: int, tx_idx: int, rel_delay: bool):
    key = (frame_idx, tx_idx, rel_delay)
    if key in _pdp_cache:
        return _pdp_cache[key]

    t = float(time_steps[frame_idx])
    pos = get_pos_at_time(t)

    # 위치/방향 업데이트
    rx.position = pos
    for name in tx_names:
        scene.transmitters[name].look_at(pos)

    # 경로 계산
    paths = solver(
        scene,
        max_depth=3,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True
    )

    # CIR 추출 (절대 지연)
    a, tau = paths.cir(out_type="tf", normalize_delays=False)

    # 너 케이스 기준:
    # a: (1, 2, 3, 128, P, 1)
    # tau: (1, 3, P)
    tau_tx = tau[0, tx_idx, :]  

    # PDP: Rx편파(2) + Tx포트(128) 합산 -> (P,)
    pdp = tf.reduce_sum(tf.abs(a[0, :, tx_idx, :, :, 0])**2, axis=[0, 1])  # (P,)

    # ---------------------------
    # padding/가짜 경로 제거
    #   - tau가 음수(-1 같은)면 padding일 가능성이 큼
    # ---------------------------
    valid = tf.math.is_finite(tau_tx) & (tau_tx >= 0)
    idx_valid = tf.where(valid)[:, 0]  # (P_valid,)

    if tf.size(idx_valid) == 0:
        # 유효 경로 없음
        df = pd.DataFrame(columns=["path_idx", "tau_ns", "pdp_db"])
        _pdp_cache[key] = (t, pos, df)
        return _pdp_cache[key]

    tau_v = tf.boolean_mask(tau_tx, valid)   # (P_valid,)
    pdp_v = tf.boolean_mask(pdp, valid)      # (P_valid,)

    # 상대지연 옵션: min τ를 0으로 이동
    if rel_delay:
        tau_v = tau_v - tf.reduce_min(tau_v)

    # 지연 기준 정렬
    order = tf.argsort(tau_v)
    tau_s = tf.gather(tau_v, order).numpy() * 1e9  # ns
    pdp_s = tf.gather(pdp_v, order).numpy()
    pdp_db = 10*np.log10(pdp_s + 1e-30)

    # 정렬된 path_idx (원래 인덱스 기준)
    path_idx_sorted = tf.gather(idx_valid, order).numpy()


    df = pd.DataFrame({
        "path_idx": path_idx_sorted.astype(int),
        "tau_ns": tau_s,
        "pdp_db": pdp_db
        })

    _pdp_cache[key] = (t, pos, df)
    return _pdp_cache[key]

def _update_plot(_=None):
    frame_idx = frame_slider.value
    tx_idx = tx_dropdown.value
    rel_delay = rel_delay_chk.value

    with out:
        out.clear_output(wait=True)

        t, pos, df = _compute_mapping_for_frame(frame_idx, tx_idx, rel_delay)

        if df.empty:
            print(f"t={t:.2f}s | frame={frame_idx} | Tx_{tx_idx+1} : 유효 경로가 없습니다 (완전 차폐/패딩 제거 후 0개).")
            print(f"pos={pos}")
            return

        # 1) 매핑 테이블 출력 (path_idx ↔ 임펄스)
        display(df)

        # 2) CIR(PDP) stem + 라벨(path_idx)
        plt.figure(figsize=(8, 3.6))
        plt.stem(df["tau_ns"].values, df["pdp_db"].values, basefmt=" ")

        # 라벨: 각 임펄스 위에 path_idx
        for x, y, pid in zip(df["tau_ns"].values, df["pdp_db"].values, df["path_idx"].values):
            plt.text(x, y, str(pid), fontsize=9, ha="center", va="bottom")

        plt.xlabel("Delay τ (ns)")
        plt.ylabel("Power (dB)")
        plt.title(f"Tx_{tx_idx+1} PDP at t={t:.2f}s | frame={frame_idx}\npos={pos}")
        plt.grid(True)
        plt.show()

# 이벤트 연결
frame_slider.observe(_update_plot, names="value")
tx_dropdown.observe(_update_plot, names="value")
rel_delay_chk.observe(_update_plot, names="value")

display(widgets.VBox([widgets.HBox([frame_slider, tx_dropdown]), rel_delay_chk]), out)
_update_plot()

Output()

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from sionna.rt.constants import InteractionType, INVALID_SHAPE

C = 299_792_458.0  # m/s

TYPE_NAME = {
    int(InteractionType.NONE): "None",
    int(InteractionType.SPECULAR): "Specular",
    int(InteractionType.DIFFUSE): "Diffuse",
    int(InteractionType.REFRACTION): "Refraction",
    int(InteractionType.DIFFRACTION): "Diffraction",
}

def paths_debug_table(paths, scene, tx_idx: int):
    syn = bool(getattr(paths, "synthetic_array", True))

    if syn:
        inter = paths.interactions.numpy()[:, 0, tx_idx, :]      # [max_depth, P]
        verts = paths.vertices.numpy()[:, 0, tx_idx, :, :]       # [max_depth, P, 3]
        objs  = paths.objects.numpy()[:, 0, tx_idx, :]           # [max_depth, P]
        tau_s = paths.tau.numpy()[0, tx_idx, :]                  # [P]
    else:
        inter = paths.interactions.numpy()[:, 0, 0, tx_idx, 0, :]
        verts = paths.vertices.numpy()[:, 0, 0, tx_idx, 0, :, :]
        objs  = paths.objects.numpy()[:, 0, 0, tx_idx, 0, :]
        tau_s = paths.tau.numpy()[0, 0, tx_idx, 0, :]

    # object_id -> object name
    id2name = {}
    for name, obj in scene.objects.items():
        oid = getattr(obj, "object_id", None)
        if oid is not None:
            id2name[int(oid)] = name

    rows = []
    P = inter.shape[-1]
    for p in range(P):
        # interaction sequence
        seq_codes = [int(x) for x in inter[:, p] if int(x) != 0]
        seq = "->".join(TYPE_NAME.get(c, str(c)) for c in seq_codes) if seq_codes else "LoS"

        # vertices
        v_list = []
        for k in range(verts.shape[0]):
            pt = verts[k, p, :]
            if not np.allclose(pt, 0):
                v_list.append(tuple(float(x) for x in pt))

        # objects
        o_list = []
        for k in range(objs.shape[0]):
            oid = int(objs[k, p])
            if oid != int(INVALID_SHAPE):
                o_list.append(id2name.get(oid, f"id:{oid}"))

        rows.append({
            "path_idx": p,
            "tau_ns": float(tau_s[p] * 1e9),
            "path_len_m": float(C * tau_s[p]),
            "interaction_seq": seq,
            "objects": o_list,
            "vertices": v_list
        })

    df = pd.DataFrame(rows).sort_values("tau_ns").reset_index(drop=True)
    df.insert(0, "arrival_rank", np.arange(len(df)))  # 도착 순서
    return df


# ---------------------------
# 표 전용 UI
# ---------------------------
out = widgets.Output()

tx_dropdown = widgets.Dropdown(
    options=[("Tx_1", 0), ("Tx_2", 1), ("Tx_3", 2)],
    value=0,
    description="TX:",
    layout=widgets.Layout(width="180px")
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_steps)-1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

# 프레임별 paths 캐시
_paths_cache = {}  # frame_idx -> (t, pos, paths)

def _get_paths(frame_idx: int):
    if frame_idx in _paths_cache:
        return _paths_cache[frame_idx]

    t = float(time_steps[frame_idx])
    pos = get_pos_at_time(t)

    rx.position = pos
    for name in tx_names:
        scene.transmitters[name].look_at(pos)

    paths = solver(
        scene,
        max_depth=3,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True
    )

    _paths_cache[frame_idx] = (t, pos, paths)
    return _paths_cache[frame_idx]

def _update(_=None):
    frame_idx = frame_slider.value
    tx_idx = tx_dropdown.value

    with out:
        out.clear_output(wait=True)

        t, pos, paths = _get_paths(frame_idx)

        df = paths_debug_table(paths, scene, tx_idx)
        df["tau_ns"] = df["tau_ns"].round(3)
        df["path_len_m"] = df["path_len_m"].round(3)

        print(f"t={t:.2f}s | frame={frame_idx} | pos={pos} | Tx_{tx_idx+1}")
        display(df)

frame_slider.observe(_update, names="value")
tx_dropdown.observe(_update, names="value")

display(widgets.VBox([widgets.HBox([frame_slider, tx_dropdown]), out]))
_update()